# eda에서 전처리로 이동할 코드


In [ ]:
# 이상치 자동 탐지
# 1. grab_col_names에서 이미 필터링된 num_cols 활용
# 순수 파생변수들만 남기기 위해 식별자 등 제외
num_features = [col for col in num_cols if col not in ['churn', 'customer_id']]

# 2. 이상치 기준선 산출 함수 (기존 로직 유지하며 가독성 개선)
def outlier_thresholds(dataframe, col_name, q1=0.05, q3=0.95):
    temp_series = dataframe[col_name].astype('float32')
    quartile1 = temp_series.quantile(q1)
    quartile3 = temp_series.quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    temp_col = dataframe[col_name].astype('float32')
    
    # 상한선이나 하한선을 벗어나는 값이 하나라도 있는지 체크
    if ((temp_col > up_limit) | (temp_col < low_limit)).any():
        return True
    return False

# 3. 우리 데이터프레임(df_analysis)으로 전체 변수 스캔 실행
print("--- 변수별 이상치(Outlier) 존재 여부 스캔 ---")
for col in num_features:
    print(f"{col: <25} : {check_outlier(df_analysis, col)}")

In [ ]:
# 다운캐스팅
# 
# 1. 문자열 -> 카테고리 변환 (우리 데이터에 존재하는 4개 컬럼)
cat_cols = ['club_member_status', 'fashion_news_frequency', 'index_group_name', 'product_group_name']
for col in cat_cols:
    df_analysis[col] = df_analysis[col].astype('category')

# 2. 정수형 타입 다운캐스팅
df_analysis['churn'] = df_analysis['churn'].astype('int8')
df_analysis['recency'] = df_analysis['recency'].astype('int32')
df_analysis['frequency'] = df_analysis['frequency'].astype('int32')
df_analysis['frequency_last_90d'] = df_analysis['frequency_last_90d'].astype('int32')
df_analysis['frequency_last_180d'] = df_analysis['frequency_last_180d'].astype('int32')
df_analysis['unique_articles'] = df_analysis['unique_articles'].astype('int32')
df_analysis['unique_product_groups'] = df_analysis['unique_product_groups'].astype('int32')

# 3. 실수형 타입 다운캐스팅 (결측치 포함 변수 및 소수점 변수)
df_analysis['age'] = df_analysis['age'].astype('float32')
df_analysis['price_mean'] = df_analysis['price_mean'].astype('float32')
df_analysis['price_sum'] = df_analysis['price_sum'].astype('float32')
df_analysis['price_max'] = df_analysis['price_max'].astype('float32')
df_analysis['avg_purchase_interval'] = df_analysis['avg_purchase_interval'].astype('float32')
df_analysis['std_purchase_interval'] = df_analysis['std_purchase_interval'].astype('float32')
df_analysis['FN'] = df_analysis['FN'].astype('float16')
df_analysis['Active'] = df_analysis['Active'].astype('float16')

# 4. 결과 확인 (메모리 용량이 대폭 줄어든 것을 볼 수 있습니다)
print(df_analysis.info())

In [ ]:
# 변수 자동 분류 및 스캔 함수
def grab_col_names(dataframe, cat_th=10, car_th=20):
    # category 타입과 object 타입을 모두 문자열 계열(범주형 대상)로 취급하도록 수정
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes in ["O", "category"]]
    
    # 숫자형(int, float) 중 고유값 개수가 임계치보다 작은 경우
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and dataframe[col].dtypes not in ["O", "category"]]
    
    # 문자열 계열 중 고유값 종류가 너무 많은 경우 (예: customer_id)
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and dataframe[col].dtypes in ["O", "category"]]
    
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # 순수 수치형 변수 정리
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes not in ["O", "category"]]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols (범주형 리스트): {len(cat_cols)}')
    print(f'num_cols (수치형 리스트): {len(num_cols)}')
    print(f'cat_but_car (고카디널리티 리스트): {len(cat_but_car)}')
    print(f'num_but_cat (수치형 내 범주성): {len(num_but_cat)}')

    return cat_cols, num_cols, cat_but_car

# 함수 실행
cat_cols, num_cols, cat_but_car = grab_col_names(df_analysis)